# Scenario 2 — Puzzletron: Hardware-Constrained Compression (~6h15 on 2x H200)

This notebook compresses Qwen3-8B to fit within a **78,000 MiB memory budget** using **Puzzletron** (heterogeneous NAS-based pruning), then distills and evaluates the result.

This is the **recommended approach** for Scenario 2. Puzzletron can directly target a memory constraint via its MIP (Mixed-Integer Programming) solver, producing a heterogeneous architecture that maximizes accuracy within the budget. For comparison, see [`scenario2_minitron.ipynb`](scenario2_minitron.ipynb).

**Pipeline:** Install → Prepare calibration data → Configure → NAS search → Evaluate → Patch → Distill → Evaluate

**Prerequisites:**
- Run [`00_prerequisites.ipynb`](00_prerequisites.ipynb) first to prepare the distillation dataset.
- Base model downloaded at `/workspace/models/Qwen3-8B`.

## 1. Install Puzzletron dependencies

Puzzletron requires additional dependencies beyond the base ModelOpt installation. Skip this step if you already ran [`scenario1_puzzletron.ipynb`](scenario1_puzzletron.ipynb) in the same container session.

In [ ]:
!cd /workspace/Model-Optimizer && \
pip install -e .[hf,puzzletron] && \
pip install -r examples/puzzletron/requirements.txt

## 2. Prepare calibration dataset

Download and format the [Nemotron-Post-Training-Dataset-v2](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2) for Puzzletron's block scoring phase. Skip if already prepared.

In [ ]:
!cd /workspace/Model-Optimizer && \
python -m modelopt.torch.puzzletron.dataset.prepare_dataset \
    --dataset_name=nvidia/Nemotron-Post-Training-Dataset-v2 \
    --output_dir=/workspace/datasets/Nemotron-Post-Training-Dataset-v2

## 3. Configure the NAS search

Puzzletron is driven by two YAML configuration files:

- **`qwen3_8b_pruneffn_memory.yaml`** — The top-level config. It sets the input model path, the calibration dataset path, the working directory, the MIP memory constraint, and the list of FFN intermediate sizes to search over (the search space). This is the file you edit for each experiment.

- **`qwen3_8b.yaml`** — The base model config. It defines the full pipeline settings: pruning strategy, scoring parameters, MIP solver settings (constraints, objective function, batch sizes for memory computation)... Most of these can be left at their defaults.

Here, the **primary constraint is memory** (78,000 MiB), not parameter count. We set:
- **`input_hf_model_path`**: path to the base Qwen3-8B model
- **`target_memory`**: 78,000 MiB — the hard memory ceiling the compressed model must fit within
- **`num_params`**: set high (8B) so it doesn't constrain — memory is the binding constraint
- **`eval_samples`**: number of samples for scoring. A higher value can produce more reliable scores and potentially a better final architecture, but scoring time scales roughly linearly with this parameter. 32 is the value we use here as a reasonable accuracy/runtime trade-off for tutorial reproducibility.

This is Puzzletron's strength: the MIP solver directly optimizes for the memory budget, accounting for both weights and KV cache per layer.

In [ ]:
!sed -i 's|input_hf_model_path: .*|input_hf_model_path: /workspace/models/Qwen3-8B|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml

!sed -i 's|target_memory: .*|target_memory: 78_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml

!sed -i 's|target_memory: .*|target_memory: 78_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

!sed -i 's|num_params: .*|num_params: 8_000_000_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

!sed -i '/^scoring:/,/^[a-z]/{s|eval_samples: .*|eval_samples: 32|}' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

### How Puzzletron computes memory footprint

To understand the 78,000 MiB target, it helps to know how Puzzletron computes memory. The total footprint is the sum of three components, computed **layer-by-layer**:

```
Total_Memory = Σ (Attention_Memory[layer] + FFN_Memory[layer]) + Non_Block_Memory
```

**Per-layer attention memory** = KV cache + attention weights:
- KV cache: `batch_size × seq_len × kv_dim × 2 × sizeof(dtype)` per layer (this is the dominant term)
- Attention weights: Wq, Wk, Wv, Wo projections + layer norm

**Per-layer FFN memory** = weight memory for the 3 linear layers (gate, up, down projections) + layer norm

**Non-block memory** = input embeddings + output LM head + final layer norm

When attention is removed from a layer (`no_op`), both the KV cache and attention weights for that layer drop to zero — this is why removing attention is far more memory-efficient than reducing FFN width.

The computation uses the inference settings from the YAML config: `batch_size=96`, `seq_len=8192` (4096 prefill + 4096 generation), `dtype=bfloat16` (2 bytes).

Let's verify by computing the memory footprint of the original Qwen3-8B:

In [ ]:
# Qwen3-8B architecture
hidden = 4096
num_heads = 32
num_kv_heads = 8
head_dim = 128
kv_dim = num_kv_heads * head_dim  # 1024
ffn_size = 12288
vocab = 151936
layers = 36
dtype_bytes = 2  # bfloat16

# Inference settings (from YAML config)
batch_size = 96
seq_len = 4096 + 4096  # prefill + generation

# --- Per-layer attention memory ---
# KV cache
kv_cache_per_layer = batch_size * seq_len * kv_dim * 2 * dtype_bytes / (1024**2)
# Attention weights: Wq + Wo + Wk + Wv + q_norm + k_norm + input_layernorm
attn_params = hidden * (num_heads * head_dim) * 2 + hidden * kv_dim * 2 + head_dim * 2 + hidden
attn_weights_per_layer = attn_params * dtype_bytes / (1024**2)
attn_total_per_layer = kv_cache_per_layer + attn_weights_per_layer

# --- Per-layer FFN memory ---
# 3 linear layers (gate, up, down) + post_attention_layernorm
ffn_params = hidden * ffn_size * 3 + hidden
ffn_per_layer = ffn_params * dtype_bytes / (1024**2)

# --- Non-block memory ---
# Input embeddings + LM head (not tied) + final RMS norm
non_block_params = vocab * hidden * 2 + hidden
non_block = non_block_params * dtype_bytes / (1024**2)

# --- Total ---
total = layers * (attn_total_per_layer + ffn_per_layer) + non_block

print(f"=== Qwen3-8B Memory Footprint (batch={batch_size}, seq={seq_len}, bf16) ===")
print(f"Per-layer KV cache:        {kv_cache_per_layer:>10.2f} MiB")
print(f"Per-layer attention weights:{attn_weights_per_layer:>9.2f} MiB")
print(f"Per-layer FFN weights:     {ffn_per_layer:>10.2f} MiB")
print(f"Per-layer total:           {attn_total_per_layer + ffn_per_layer:>10.2f} MiB")
print(
    f"All {layers} layers:            {layers * (attn_total_per_layer + ffn_per_layer):>10.2f} MiB"
)
print(f"Non-block (embed + LM head):{non_block:>9.2f} MiB")
print()
print(f"TOTAL:                     {total:>10.2f} MiB ({total / 1024:.2f} GiB)")
print()
print(f"KV cache share: {layers * kv_cache_per_layer / total * 100:.1f}% of total memory")
print(
    f"Target budget:  78,000 MiB -> need to reduce by {total - 78000:.0f} MiB ({(total - 78000) / total * 100:.1f}%)"
)

**Expected output:**

| Component | Per Layer | All 36 Layers | Share |
|---|---|---|---|
| KV cache | 3,072 MiB | 110,592 MiB | 87.6% |
| Attention weights | 80 MiB | 2,880 MiB | 2.3% |
| FFN weights | 288 MiB | 10,368 MiB | 8.2% |
| Embeddings + LM head | — | 2,374 MiB | 1.9% |
| **Total** | **3,440 MiB** | **126,215 MiB (123.26 GiB)** | **100%** |

The KV cache alone accounts for nearly 88% of the total memory. This is why Puzzletron's ability to remove attention (and its KV cache) from selected layers is so effective for meeting a memory target. To reach 78,000 MiB, we need to cut ~48,215 MiB (38.2%).

## 4. Run Puzzletron NAS search (Longest step: 5 hours at first run)

This is the core Puzzletron pipeline. It runs the full 8-step process:
1. Convert the model to Puzzletron's heterogeneous format
2. Score pruning activations across all layers
3. Generate pruned checkpoint variants at different FFN sizes
4. Build a replacement library of all per-layer candidates
5. Calculate memory/parameter stats for each candidate
6. Score each replacement's quality — longest step
7. Run MIP optimization to find the best architecture within the memory constraint
8. Assemble the final heterogeneous model

### Search space

For each of the 36 Transformer layers, the MIP solver can choose from the following options:

**FFN block:** keep the original (intermediate_size=12288), replace with a pruned variant at one of the candidate sizes `[2560, 5120, 7424, 9984]`, or remove it entirely (`no_op`). That's **6 FFN options per layer**.

**Attention block:** keep the original (GQA with 8 KV heads) or remove it entirely (`no_op`). That's **2 attention options per layer**.

Combined, each layer has up to **12 possible configurations** (6 FFN × 2 attention). Across 36 layers, the theoretical search space is 12³⁶ — far too large to enumerate. The MIP solver efficiently finds the optimal combination by formulating it as a constrained optimization problem.

The MIP solver will find the optimal heterogeneous architecture that fits within 78,000 MiB. Unlike Scenario 1 (where the model only reduced FFN widths), here the solver is expected to also **remove attention from multiple layers** — since KV cache is the dominant memory consumer, removing attention from a layer saves far more memory than thinning its FFN.

In [ ]:
# Remove if already exists from a previous run
!rm -f /workspace/puzzle_dir/subblock_stats.json
!cd /workspace/Model-Optimizer && \
torchrun --nproc_per_node 1 \
    examples/puzzletron/main.py \
    --config examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml \
    2>&1 | tee /workspace/puzzletron_qwen3_78k.log

## 5. Evaluate pruned model (before distillation)

At ~38% memory compression, the model is expected to be near-random before distillation. This is normal — distillation will recover significant accuracy.

> **Note:** The `sed` command below fixes a dtype formatting issue in the generated config.

In [ ]:
!sed -i 's/"torch\.bfloat16"/"bfloat16"/g' \
    /workspace/puzzle_dir/mip/puzzle_solutions/target_memory_78000MiB-num_params_8G/solutions--checkpoints/solution_0/config.json

!cd /workspace/Model-Optimizer && \
python examples/llm_eval/lm_eval_hf.py \
    --model hf \
    --model_args pretrained=/workspace/puzzle_dir/mip/puzzle_solutions/target_memory_78000MiB-num_params_8G/solutions--checkpoints/solution_0/,dtype=bfloat16,parallelize=True \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

## 6. Distill

Distill the heterogeneous model against the original Qwen3-8B teacher. Same recipe: 100 iterations on [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1). At this compression level, distillation is critical — it transforms the model from near-random to functional.

The `distill.py` script handles both distillation and automatic export to HuggingFace format.

Launch TensorBoard to monitor the distillation loss in real time. Open http://localhost:6006 in your browser once the distillation cell is running.

> **Tip:** In the TensorBoard settings (top-right gear icon), check **"Reload data"** so the charts update automatically as training progresses.

In [ ]:
import subprocess

subprocess.Popen(
    [
        "tensorboard",
        "--logdir",
        "/workspace/output/distill_output_puzzle_78k/tb_logs",
        "--port",
        "6006",
    ]
)
print("TensorBoard started at http://localhost:6006")

Now, let's run the distillation.
> **Expected runtime: ~20-30 minutes on 2x H200.**

In [ ]:
!torchrun --nproc_per_node=2 \
    /workspace/Model-Optimizer/examples/megatron_bridge/distill.py \
    --student_hf_path /workspace/puzzle_dir/mip/puzzle_solutions/target_memory_78000MiB-num_params_8G/solutions--checkpoints/solution_0 \
    --teacher_hf_path /workspace/models/Qwen3-8B \
    --data_paths 1.0 /workspace/datasets/wikitext-103-v1/wikitext-train_text \
    --output_dir /workspace/output/distill_output_puzzle_78k \
    --hf_export_path /workspace/output/distilled_Qwen3-8B-Puzzle-78k \
    --student_hf_model Qwen/Qwen3-8B \
    --seq_length 4096 \
    --tp_size 2 \
    --pp_size 1 \
    --mbs 1 \
    --gbs 4 \
    --train_iters 100 \
    --lr 0.0001 \
    --min_lr 1e-05 \
    --lr_warmup_iters 10 \
    --eval_interval 10 \
    --eval_iters 10 \
    --log_interval 1

Finally, kill tensorboard:

In [ ]:
subprocess.run(["pkill", "-f", "tensorboard"])

## 7. Evaluate distilled model

Compare with the Minitron result at the same memory budget (see [`scenario2_minitron.ipynb`](scenario2_minitron.ipynb)).

**Expected results on Qwen3-8B:**

| Model | Memory | MMLU (5-shot) | % of Teacher |
|---|---|---|---|
| Qwen3-8B (teacher) | 126,215 MiB | 0.7493 | 100% |
| Puzzletron 78k — pruned | 77,992 MiB | 0.2752 | 36.7% |
| **Puzzletron 78k — distilled** | **77,992 MiB** | **0.5613** | **74.9%** |
| Minitron 22L — distilled | 78,054 MiB | 0.4620 | 61.7% |

In [ ]:
!cd /workspace/Model-Optimizer && \
python examples/llm_eval/lm_eval_hf.py \
    --model hf \
    --model_args pretrained=/workspace/output/distilled_Qwen3-8B-Puzzle-78k,dtype=bfloat16,parallelize=True \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

## 8. Bonus: Memory Sweep

Puzzletron supports a **MIP (Mixed-Integer Programming) sweep mode** that lets you explore multiple memory compression rates in a single run. Instead of running the full pipeline for each target, the sweep reuses the scoring and replacement library from Steps 1–6 and only re-runs the MIP solver at each compression rate making it very fast.

This produces a CSV with accuracy and memory metrics for each configuration, allowing you to map out the accuracy-memory trade-off curve and find the right operating point for your deployment.

### Enable sweep mode

We add the sweep configuration to the YAML and run with `--mip-only` (skips the scoring steps that were already completed in Step 4):

In [ ]:
import yaml

config_path = "/workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

# Add sweep configuration
config["mip"]["sweep"] = {
    "enabled": True,
    "memory_compression_rates": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "output_csv": "/workspace/puzzle_dir/mip_sweep_results.csv",
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("Sweep config added. Compression rates: [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]")

### Run the sweep

In [ ]:
!cd /workspace/Model-Optimizer && \
torchrun --nproc_per_node 1 \
    examples/puzzletron/main.py \
    --config examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml \
    --mip-only \
    2>&1 | tee /workspace/puzzletron_sweep.log | grep "Puzzletron Progress"

Here is an example of what the accuracy-memory trade-off curve looks like when we ran this sweep (MMLU accuracy for Qwen3-8B w.r.t. memory compression rate):

![Puzzletron Memory Sweep](memory_sweep.png)